# 문서를 넣으면 갱신되는 지식베이스 — 파이프라인

**「코랩으로 만드는 업무 챗봇」 16장 예제** — 문서를 투입구에 넣으면 보관 → 검증 → 임베딩 → 적재까지 한 번에 도는 **파이프라인(DAG)** 을 만들고, 다시 돌려도 안전한지 실측으로 확인합니다.

- 걸리는 시간: 약 50분
- 키: 필요 없습니다(전부 키 없이 끝까지 돕니다).

### 시작하는 방법
1. 「Drive로 복사」를 누른 뒤 위에서부터 한 칸씩 실행합니다(칸 왼쪽 ▶ 또는 Shift+Enter).
2. 「준비」 아래의 칸들은 처음 한 번만 실행합니다. 설치는 1~2분 걸립니다.
3. 예제마다 **확인 포인트**와 **바꿔 보기**가 있습니다. 확인 포인트의 값이 화면과 같은지 보고, 바꿔 보기는 직접 고쳐 다시 실행합니다.

### 이 노트북의 환경
실제 서비스에서는 Airflow 를 서버에 따로 띄웁니다. 이 노트북은 같은 Airflow(3.3.2)를 코랩 안 가상환경 하나에 넣어, 설치 걱정 없이 파이프라인에만 집중합니다. 코랩 런타임이 끊기면 처음부터 다시 실행하면 됩니다.

### 예제 구성
| 번호 | 예제 |
|---|---|
| 1 | 재료 준비 — 문서 · 도구, 그리고 빈 지식베이스(대조군) |
| 2 | 파이프라인 DAG 를 만들어 돌리기 |
| 3 | 멱등성 — 두 번 돌려도 조각 수가 같은가 |
| 4 | 새 문서 투입 — 넣기 전과 뒤의 검색 점수 |
| 5 | 일부러 나쁜 문서 넣기 — validate 가 막는가 |
| 6 | 문서를 지우면 — 구멍을 찾고 prune 으로 메우기 |

## 준비

In [ ]:
# 준비 1 — Airflow 3.3.2 설치 (1~2분, 처음 한 번만)
import os, subprocess, sys, time

os.environ["TZ"] = "Asia/Seoul"          # 코랩 시계는 UTC — 화면의 시각을 한국 기준으로
time.tzset()

AF_VER = "3.3.2"                          # 이 책이 확인한 판
PY = f"{sys.version_info.major}.{sys.version_info.minor}"
HOME = "/content"
VENV = f"{HOME}/af"                       # 코랩 꾸러미와 섞이지 않게 가상환경에 설치합니다
PORT = 18080                              # 기본값은 8080 — 코랩이 8080 을 이미 써서 다른 번호를 씁니다

os.environ.update({
    "AIRFLOW_HOME": f"{HOME}/airflow",
    "AIRFLOW__CORE__DAGS_FOLDER": f"{HOME}/dags",      # DAG 파일을 두는 폴더
    "AIRFLOW__CORE__LOAD_EXAMPLES": "False",
    "AIRFLOW__CORE__PARALLELISM": "4",
    "AIRFLOW__API__PORT": str(PORT),
    "AIRFLOW__API__BASE_URL": f"http://localhost:{PORT}",
    "AIRFLOW__CORE__EXECUTION_API_SERVER_URL": f"http://localhost:{PORT}/execution/",
    "NO_COLOR": "1",                                    # 로그에 색 코드가 섞이지 않게
    "PATH": f"{VENV}/bin:" + os.environ["PATH"],
})
for d in ("dags", "data"):
    os.makedirs(f"{HOME}/{d}", exist_ok=True)

if not os.path.exists(f"{VENV}/bin/airflow"):
    print("설치 중입니다 …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True, capture_output=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "-q", VENV, "--python", sys.executable], check=True)
    r = subprocess.run(
        [sys.executable, "-m", "uv", "pip", "install", "-q", "--python", f"{VENV}/bin/python",
         f"apache-airflow=={AF_VER}", "graphviz", 'fastembed', 'qdrant-client',
         "--constraint", f"https://raw.githubusercontent.com/apache/airflow/constraints-{AF_VER}/constraints-{PY}.txt"],
        capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise SystemExit("설치하지 못했습니다. 이 칸을 한 번 더 실행해 주세요.")

r = subprocess.run("airflow db migrate", shell=True, capture_output=True, text=True)   # 메타데이터 DB(SQLite) 만들기
v = subprocess.run("airflow version", shell=True, capture_output=True, text=True).stdout.strip().splitlines()[-1]
print("설치 확인 · Airflow", v, "· 파이썬", PY)
print("DAG 폴더 ·", os.environ["AIRFLOW__CORE__DAGS_FOLDER"])

In [ ]:
# 준비 2 — 이 노트북에서 쓰는 도구 셋 (처음 한 번만)
import json, re
from datetime import datetime, timezone
from IPython.display import Image, display

_LOG = re.compile(r"^\d{4}-\d\d-\d\dT\S+ \[")
_ANSI = re.compile(r"\x1b\[[0-9;]*m")


def _sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return _ANSI.sub("", r.stdout + r.stderr)
_NOISE = ("graphviz", "You might need to install", "Please be reminded", "DagRun failed")


def af(args, show=True):
    """Airflow 명령을 부릅니다 — af("dags list") = airflow dags list. 로그 줄은 빼고 결과만 보여 줍니다."""
    keep = [l for l in _sh("airflow " + args).splitlines()
            if l.strip() and not _LOG.match(l) and not any(n in l for n in _NOISE)]
    text = "\n".join(keep)
    if show:
        print(text)
        return None                         # 화면에 찍었으면 돌려주지 않습니다(칸 아래에 한 번 더 나오지 않게)
    return text


def graph(dag_id):
    """DAG 모양을 그림으로 — 웹 UI 의 Graph 화면과 같은 것입니다."""
    png = f"{HOME}/data/{dag_id}.png"
    if os.path.exists(png):
        os.remove(png)
    af(f"dags show {dag_id} --save {png}", show=False)
    if not os.path.exists(png):             # 아직 등록 전인 새 파일이면 먼저 읽게 한 뒤 다시
        af("dags reserialize", show=False)
        af(f"dags show {dag_id} --save {png}", show=False)
    if os.path.exists(png):
        display(Image(png))
    else:                                   # 그림 도구가 없으면 화살표만 글자로
        print("\n".join(l.strip() for l in af(f"dags show {dag_id}", show=False).splitlines() if "->" in l))


def _utc(s):
    return datetime.fromisoformat(s).replace(tzinfo=timezone.utc)      # DB 의 시각은 UTC


def _kst(s):
    return _utc(s).astimezone().strftime("%H:%M:%S.%f")[:-3] if s else "-"


def run_dag(dag_id, conf=None):
    """DAG 를 한 번 끝까지 돌리고(airflow dags test), 태스크가 찍은 글과 태스크별 상태를 보여 줍니다."""
    cmd = f"airflow dags test {dag_id}" + (f" -c '{json.dumps(conf)}'" if conf else "")
    print(f"$ {cmd}\n실행 중입니다 …\n")
    out = _sh(cmd)
    task, tb = "-", []

    def flush():
        if tb:
            print(f"  [{task}] 오류 → {tb[-1].strip()}")
            tb.clear()

    for l in out.splitlines():
        m = re.search(r"\[DAG TEST\] starting task_id=(\S+)", l)
        if m:
            flush()
            task = m.group(1)
            continue
        if _LOG.match(l):
            flush()
            m = re.search(r"Done\. Returned value was: (.*?)\s+\[airflow", l)
            if m and m.group(1) != "None":
                print(f"  [{task}] 돌려준 값(XCom) → {m.group(1)}")
            m = re.search(r"Branch into (\S+)", l)
            if m:
                print(f"  [{task}] 고른 갈래 → {m.group(1)}")
            if "Skipping downstream tasks" in l and "ShortCircuitOperator" in l:
                print(f"  [{task}] False 를 돌려줘 뒤 태스크를 건너뜁니다")
            continue
        if not l.strip() or any(n in l for n in _NOISE):
            continue
        if tb or l.startswith("Traceback"):
            tb.append(l)
        else:
            print(f"  [{task}] {l}")
    flush()

    m = re.search(r"run_id=(manual__[^\s,]+)", out)
    if not m:                               # DAG 를 읽지 못한 경우 — 원인을 그대로 보여 줍니다
        print("\n".join(l for l in out.splitlines() if not _LOG.match(l))[-1500:])
        print("\nDAG 를 돌리지 못했습니다. 위 오류와 af(\"dags list-import-errors\") 를 확인해 주세요.")
        return {}
    import sqlite3                           # 태스크 상태는 메타데이터 DB 에서 바로 읽습니다(웹 UI 의 Grid 가 보는 곳)
    db = sqlite3.connect(f"{HOME}/airflow/airflow.db")
    rows = [dict(zip(("task_id", "state", "start_date", "end_date", "try_number"), r)) for r in db.execute(
        "select task_id, state, start_date, end_date, try_number from task_instance where dag_id=? and run_id=?",
        (dag_id, m.group(1)))]
    db.close()
    rows.sort(key=lambda x: x.get("start_date") or "9")
    print(f"\n{'태스크':<16}{'상태':<18}{'시도':<5}{'시작(한국 시각)':<16}걸린 시간")
    for x in rows:
        sec = ""
        if x.get("start_date") and x.get("end_date"):
            sec = f"{(_utc(x['end_date']) - _utc(x['start_date'])).total_seconds():.2f}초"
        print(f"{x['task_id']:<19}{(x['state'] or '-'):<20}{x['try_number']:<7}{_kst(x.get('start_date')):<20}{sec}")
    state = re.search(r"DagRun Finished:.*?state=(\w+)", out)
    print(f"\n실행 전체 상태 → {state.group(1) if state else '?'}")
    return {x["task_id"]: x["state"] for x in rows}


print("도구 준비 완료 · af( ) · graph( ) · run_dag( )")

In [ ]:
import shlex

for d in ("inbox", "lake/raw-docs"):
    os.makedirs(f"{HOME}/{d}", exist_ok=True)


def kb(cmd, *args):
    """kb.py 를 부릅니다 — kb("stat") · kb("search", "질문") · kb("ask", "질문") · kb("reset")"""
    line = f"{VENV}/bin/python {HOME}/dags/kb.py {cmd} " + " ".join(shlex.quote(str(a)) for a in args)
    out = _sh(line)
    keep = [l for l in out.splitlines() if l.strip() and "Fetching" not in l and "Warning" not in l and "warn" not in l]
    print("\n".join(keep))


def put(name, text):
    """투입구(inbox)에 문서를 넣습니다"""
    open(f"{HOME}/inbox/{name}", "w").write(text)
    print(f"투입구에 넣음 · {name} ({len(text)}자)")


print("도구 준비 완료 · kb( ) · put( )")

## 예제 1 · 재료 준비 — 문서 · 도구, 그리고 빈 지식베이스(대조군)

저장하는 곳은 모두 코랩 안입니다. 실제 서비스로 옮길 때 무엇으로 바뀌는지 함께 적었습니다.

| 단계 | 이 노트북 | 실제 서비스라면 |
|---|---|---|
| 투입구 | `/content/inbox/` 폴더 | 공유 폴더 · 올리기 화면 |
| Lake(원본 보관) | `/content/lake/raw-docs/` 폴더 | 오브젝트 저장소(S3 · MinIO) |
| 임베딩 | 코랩 안에서 도는 다국어 모델(384차원 · 키 없음) | 임베딩 API · 사내 모델 |
| VectorDB | Qdrant, 파일 저장 방식 `lecture_kb` | Qdrant 서버 |

먼저 문서 세 개를 투입구에 넣고, 지식베이스 도구 `kb.py` 를 만듭니다. `kb.py` 는 DAG 가 불러 쓰는 모듈이라 `dags/` 폴더에 둡니다.

In [ ]:
# 실습 문서 세 개 — 과정에서 다룬 도구의 짧은 안내문입니다
DOCS = {
    "airflow_ops.txt": """Airflow 운영 안내

Airflow 는 여러 단계로 된 작업을 정해진 순서와 시각에 실행하는 오케스트레이션 도구입니다. 작업의 순서도는 DAG 라고 부르고 파이썬 파일로 적습니다.

스케줄러가 멈추면 웹 화면은 열리지만 실행이 queued 상태에 머뭅니다. 이때는 docker compose ps 로 스케줄러 컨테이너의 상태를 보고 docker compose logs airflow-scheduler 로 원인을 확인합니다.

태스크가 실패하면 retries 에 적은 횟수만큼 다시 시도합니다. 그래도 실패하면 뒤 태스크는 upstream_failed 가 되어 시작하지 않습니다. 원인을 고친 뒤 실패한 태스크만 Clear 하면 그 자리부터 이어서 실행됩니다.

새 DAG 파일은 dags 폴더에 넣으면 dag-processor 가 읽어 등록합니다. 목록에 보이지 않으면 airflow dags list-import-errors 로 문법 오류가 있는지 먼저 확인합니다.
""",
    "minio_guide.txt": """MinIO 보관 안내

MinIO 는 S3 와 호환되는 오브젝트 저장소입니다. 우리 과정에서는 데이터 레이크의 저장소로 쓰며, 원본 문서를 고치지 않고 그대로 보관합니다.

문서는 버킷이라는 큰 통에 담습니다. 원본 문서는 raw-docs 버킷에 올리고, 같은 이름으로 다시 올리면 이전 파일을 덮어씁니다. 그래서 같은 문서를 여러 번 올려도 파일은 하나만 남습니다.

버킷의 내용은 mc ls 명령으로 확인합니다. 웹 콘솔은 9001 포트에서 열리고 프로그램은 9000 포트의 API 로 접속합니다.

원본을 Lake 에 먼저 보관해 두면 임베딩 방법이나 조각 크기를 바꾸고 싶을 때 원본에서 다시 만들 수 있습니다.
""",
    "qdrant_guide.txt": """Qdrant 검색 안내

Qdrant 는 벡터 데이터베이스입니다. 문서를 작은 조각으로 자르고 조각마다 임베딩 벡터를 만들어 넣어 두면, 질문과 뜻이 가까운 조각을 찾아 줍니다.

조각을 넣는 단위는 포인트이고 포인트의 묶음을 컬렉션이라고 부릅니다. 컬렉션을 만들 때 정한 벡터 차원과 넣는 벡터의 차원이 다르면 오류가 납니다.

포인트의 id 를 문서 이름과 조각 순번으로 만들면 같은 문서를 다시 넣어도 같은 자리에 덮어씁니다. 무작위 id 를 쓰면 넣을 때마다 조각이 중복으로 쌓입니다.

검색 결과에는 0 에서 1 사이의 유사도 점수가 붙습니다. 점수가 낮은 결과는 질문과 관계없는 조각이므로 기준 점수보다 낮으면 답하지 않는 것이 안전합니다.
""",
}

for name, text in DOCS.items():
    put(name, text)

In [ ]:
%%writefile /content/dags/kb.py
# kb.py — 지식베이스 도구
import sys
import uuid
from pathlib import Path

LAKE = Path("/content/lake/raw-docs")       # Lake — 원본 문서를 그대로 두는 곳
QDRANT = "/content/qdrant"                  # VectorDB — Qdrant(파일로 저장)
COLLECTION = "lecture_kb"
MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # 384차원 · 키 없이 코랩 안에서 계산
DIM = 384
THRESHOLD = 0.40                            # 이 점수보다 낮으면 「찾지 못했습니다」 — kb_02 에서 재서 정한 값

_model = None


def embed(texts):
    global _model
    if _model is None:
        from fastembed import TextEmbedding
        _model = TextEmbedding(MODEL, cache_dir="/content/fastembed")
    return [v.tolist() for v in _model.embed(list(texts))]


def chunk(text, size=200, overlap=40):
    """문단을 이어 붙여 size 글자쯤 되면 한 조각 — 조각 끝 overlap 글자는 다음 조각 앞에 다시 넣습니다."""
    chunks, buf = [], ""
    for para in [p.strip() for p in text.split("\n\n") if p.strip()]:
        if buf and len(buf) + len(para) > size:
            chunks.append(buf)
            buf = buf[-overlap:]
        buf = (buf + " " + para).strip()
    if buf:
        chunks.append(buf)
    return chunks


def client():
    from qdrant_client import QdrantClient, models
    c = QdrantClient(path=QDRANT)
    if not c.collection_exists(COLLECTION):
        c.create_collection(COLLECTION, vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE))
    return c


def _of(source):
    from qdrant_client import models
    return models.Filter(must=[models.FieldCondition(key="source", match=models.MatchValue(value=source))])


def remove_doc(c, source):
    from qdrant_client import models
    c.delete(COLLECTION, points_selector=models.FilterSelector(filter=_of(source)))


def index_doc(c, source, text, extra=None):
    """문서 하나를 넣습니다 — 먼저 그 문서의 조각을 지우고, 「문서 이름#순번」으로 만든 id 로 다시 넣습니다(멱등)."""
    from qdrant_client import models
    remove_doc(c, source)
    chunks = chunk(text)
    points = [models.PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{source}#{i}")), vector=v,
                                 payload={"source": source, "no": i, "text": t, **(extra or {})})
              for i, (t, v) in enumerate(zip(chunks, embed(chunks)))]
    c.upsert(COLLECTION, points=points)
    return len(points)


def stat(c):
    """문서별 조각 수"""
    out, offset = {}, None
    while True:
        points, offset = c.scroll(COLLECTION, limit=256, offset=offset, with_payload=True)
        for p in points:
            out[p.payload["source"]] = out.get(p.payload["source"], 0) + 1
        if offset is None:
            return dict(sorted(out.items()))


def search(question, k=3):
    c = client()
    try:
        hits = c.query_points(COLLECTION, query=embed([question])[0], limit=k, with_payload=True).points
        return [(round(h.score, 3), h.payload["source"], h.payload["text"]) for h in hits]
    finally:
        c.close()


def answer(question, threshold=THRESHOLD):
    """가장 가까운 조각의 점수가 기준보다 낮으면 답하지 않습니다."""
    hits = search(question)
    if not hits or hits[0][0] < threshold:
        top = hits[0][0] if hits else 0
        return f"찾지 못했습니다 (가장 가까운 조각의 점수 {top} < 기준 {threshold})"
    score, source, text = hits[0]
    return f"[근거 {source} · 점수 {score}]\n{text}"


if __name__ == "__main__":                  # 터미널에서: python kb.py stat | search "질문" | ask "질문" | reset
    cmd = sys.argv[1] if len(sys.argv) > 1 else "stat"
    if cmd == "reset":
        import shutil
        shutil.rmtree(QDRANT, ignore_errors=True)
        print("지식베이스를 비웠습니다")
    elif cmd == "stat":
        c = client()
        s = stat(c)
        c.close()
        print(f"조각 {sum(s.values())}개 · 문서 {len(s)}개")
        for name, n in s.items():
            print(f"  {name:20s} {n}조각")
    elif cmd == "search":
        for score, source, text in search(sys.argv[2]):
            print(f"  {score:<6} {source:18s} {text[:46]}…")
    elif cmd == "ask":
        print(answer(sys.argv[2], float(sys.argv[3]) if len(sys.argv) > 3 else THRESHOLD))

파이프라인을 돌리기 **전에** 먼저 물어봅니다. 이 결과가 대조군입니다. (첫 실행은 임베딩 모델을 내려받느라 30초쯤 걸립니다.)

In [ ]:
kb("stat")
print()
kb("ask", "스케줄러가 멈추면 어떻게 되나요?")

***확인 포인트***: 조각 0개 · 문서 0개이고, 질문에는 「찾지 못했습니다 (가장 가까운 조각의 점수 0 < 기준 0.4)」가 나옵니다. 문서는 투입구에 있지만 아직 아무도 지식베이스로 옮기지 않았습니다.

**바꿔 보기**
- `!ls -la /content/inbox /content/lake/raw-docs` 로 두 폴더를 봅니다. 투입구에는 세 파일이 있고 Lake 는 비어 있습니다.

## 예제 2 · 파이프라인 DAG 를 만들어 돌리기

일을 네 태스크로 나눕니다. 태스크마다 「다시 돌려도 안전한가(멱등)」를 함께 적습니다.

| 태스크 | 하는 일 | 멱등한가 |
|---|---|---|
| `collect` | 투입구의 문서를 Lake 로 올린다 | 같은 이름에 덮어씀 → 예 |
| `validate` | Lake 의 문서가 너무 짧지 않은지 검사, 미달이면 실패 | 읽기만 → 예 |
| `build_index` | 문서를 잘라 임베딩해 `lecture_kb` 에 넣는다 | 문서마다 지우고 다시 넣기 → 예 |
| `report` | 지식베이스의 조각 수를 남긴다 | 읽기만 → 예 |

In [ ]:
%%writefile /content/dags/rag_pipeline.py
# rag_pipeline.py — 수집 → 검증 → 임베딩 · 적재 → 보고
import shutil
from datetime import datetime
from pathlib import Path

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator

import kb                                   # 같은 dags/ 폴더의 kb.py

INBOX = Path("/content/inbox")              # 투입구 — 새 문서를 넣는 곳
MIN_CHARS = 50                              # 이보다 짧은 문서는 통과시키지 않습니다


def collect():
    """투입구의 문서를 Lake 로 올립니다 — 같은 이름이면 덮어씁니다(멱등)."""
    kb.LAKE.mkdir(parents=True, exist_ok=True)
    names = sorted(p.name for p in INBOX.glob("*.txt"))
    for name in names:
        shutil.copyfile(INBOX / name, kb.LAKE / name)
    print(f"Lake 에 올린 문서 {len(names)}개 → {names}")
    return names


def validate():
    """Lake 의 문서가 비었거나 너무 짧지 않은지 봅니다 — 미달이면 여기서 멈춥니다."""
    bad = []
    for p in sorted(kb.LAKE.glob("*.txt")):
        n = len(p.read_text().strip())
        print(f"{p.name:20s} {n}자")
        if n < MIN_CHARS:
            bad.append(f"{p.name}({n}자)")
    if bad:
        raise ValueError(f"너무 짧은 문서가 있습니다 → {bad}")


def build_index():
    """문서를 잘라 임베딩해 지식베이스에 넣습니다 — 문서마다 지우고 다시 넣습니다(멱등)."""
    c = kb.client()
    try:
        total = 0
        for p in sorted(kb.LAKE.glob("*.txt")):
            n = kb.index_doc(c, p.name, p.read_text())
            print(f"{p.name:20s} {n}조각")
            total += n
        return total
    finally:
        c.close()


def report(ti):
    c = kb.client()
    try:
        s = kb.stat(c)
    finally:
        c.close()
    print(f"이번에 넣은 조각 {ti.xcom_pull(task_ids='build_index')}개 · 지식베이스 전체 {sum(s.values())}개 · 문서 {len(s)}개")


with DAG(
    dag_id="rag_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["week8", "rag"],
) as dag:
    t = {f.__name__: PythonOperator(task_id=f.__name__, python_callable=f)
         for f in (collect, validate, build_index, report)}

    t["collect"] >> t["validate"] >> t["build_index"] >> t["report"]

In [ ]:
s = run_dag("rag_pipeline")
graph("rag_pipeline")

In [ ]:
kb("stat")
print()
kb("ask", "스케줄러가 멈추면 어떻게 되나요?")
print()
kb("search", "원본 문서는 어디에 보관하나요?")

***확인 포인트***: 네 태스크가 모두 success 이고 `report` 가 「지식베이스 전체 10개 · 문서 3개」를 찍습니다(4 · 3 · 3조각).
같은 질문에 이번에는 **`[근거 airflow_ops.txt · 점수 0.476]`** 과 함께 스케줄러 문단이 나옵니다. 예제 1 의 대조군과 비교하면 달라진 것은 파이프라인을 한 번 돌린 것뿐입니다.

`search` 는 가까운 조각 셋을 점수와 함께 보여 줍니다. 「원본 문서 보관」 질문은 `minio_guide.txt` 조각이 0.6 으로 1위입니다.

**바꿔 보기**
- `kb.py` 의 `chunk` 는 200자쯤에서 자릅니다. `rag_pipeline.py` 의 `kb.index_doc(c, p.name, p.read_text())` 는 그대로 두고 `kb.py` 에서 `size=200` 을 `100` 으로 바꿔 다시 돌리면 조각 수는 몇 개가 되나요? 같은 질문의 점수는 오르나요, 내리나요? (확인한 뒤 200 으로 되돌리고 한 번 더 돌립니다.)

## 예제 3 · 멱등성 — 두 번 돌려도 조각 수가 같은가

스케줄에 따라 매일 돌고, 실패하면 재시도하고, 사람이 Clear 로 다시 돌립니다. 그때마다 조각이 늘어나면 같은 문단이 검색 결과를 차지하게 됩니다.
아무것도 바꾸지 않고 한 번 더 돌려 봅니다.

In [ ]:
s = run_dag("rag_pipeline")
print()
kb("stat")

***확인 포인트***: 한 번 더 돌린 뒤에도 **조각 10개 · 문서 3개 그대로**입니다. 두 가지가 이것을 지킵니다.

- `collect` 는 같은 이름의 파일에 덮어씁니다.
- `index_doc` 은 그 문서의 조각을 먼저 지우고, id 를 `문서 이름#순번` 으로 만들어 넣습니다(`uuid5`).

**바꿔 보기**
- 일부러 깨뜨려 봅니다. `kb.py` 의 `index_doc` 에서 `remove_doc(c, source)` 줄을 주석 처리하고, id 를 `str(uuid.uuid4())`(무작위)로 바꾼 뒤 두 번 돌립니다. 조각 수가 어떻게 되나요? 확인한 뒤 `kb("reset")` 으로 비우고 코드를 되돌려 다시 돌립니다.

## 예제 4 · 새 문서 투입 — 넣기 전과 뒤의 검색 점수

지식베이스에 없는 주제를 물어봅니다. 그다음 그 주제의 문서를 투입구에 넣고 파이프라인을 돌린 뒤 같은 질문을 다시 합니다.

In [ ]:
print("── 넣기 전")
kb("search", "레이크하우스가 뭔가요?")
print()
kb("ask", "레이크하우스가 뭔가요?")

In [ ]:
LAKEHOUSE = """레이크하우스 안내

레이크하우스는 데이터 레이크와 데이터 웨어하우스의 장점을 합친 구조입니다. 값싼 오브젝트 저장소에 파일을 두면서도 표처럼 다룰 수 있습니다.

우리 과정에서는 MinIO 위에 Iceberg 표 형식을 얹어 레이크하우스를 만들었습니다. Iceberg 는 어떤 파일이 표에 속하는지와 표의 변경 이력을 기록합니다.

같은 Iceberg 표를 Spark 와 Trino 두 엔진으로 읽을 수 있습니다. Spark 로 적재하고 Trino 로 빠르게 조회하는 식으로 역할을 나눕니다.

스냅샷이 남기 때문에 과거 시점의 데이터를 다시 조회할 수 있고, 잘못 적재했을 때 이전 스냅샷으로 되돌릴 수 있습니다.
"""

put("lakehouse.txt", LAKEHOUSE)

s = run_dag("rag_pipeline")

In [ ]:
print("── 넣은 뒤")
kb("search", "레이크하우스가 뭔가요?")
print()
kb("ask", "레이크하우스가 뭔가요?")
print()
kb("stat")

***확인 포인트***: 넣기 전에는 `minio_guide.txt` 조각이 0.372 로 1위였습니다. 「데이터 레이크」라는 낱말이 겹쳐 조금 가깝게 나온 것인데, 기준(0.40)보다 낮아 「찾지 못했습니다」로 답했습니다. 기준이 없었다면 MinIO 문단을 근거로 엉뚱한 답을 했을 자리입니다.
넣은 뒤에는 `lakehouse.txt` 조각이 **0.622** 로 1위에 오르고 답이 나옵니다. 조각은 10 → 12개, 문서는 3 → 4개입니다.

사람이 한 일은 **파일 하나를 투입구에 넣은 것**뿐입니다. 올리고, 검사하고, 자르고, 임베딩하고, 넣는 일은 DAG 가 했습니다. 스케줄을 `@daily` 로 바꾸면 그 「돌리기」마저 사람이 하지 않게 됩니다 — kb_03 노트북에서 합니다.

**바꿔 보기**
- 내 업무 문서를 하나 넣어 봅니다. `put("my_doc.txt", """…내용…""")` 으로 넣고 돌린 뒤 그 문서에만 있는 내용을 질문합니다.

## 예제 5 · 일부러 나쁜 문서 넣기 — validate 가 막는가

내용이 거의 없는 문서를 투입구에 넣습니다. 이런 문서가 지식베이스에 들어가면 짧은 조각이 엉뚱한 질문에 걸려 나옵니다.

In [ ]:
put("empty.txt", "작성 중")

s = run_dag("rag_pipeline")
print()
kb("stat")

***확인 포인트***: `collect` success · **`validate` failed** — 「너무 짧은 문서가 있습니다 → ['empty.txt(4자)']」 · `build_index` 와 `report` 는 **upstream_failed** 입니다.
지식베이스는 **12개 · 문서 4개 그대로**입니다. 나쁜 문서가 임베딩 단계까지 내려가지 못했습니다.

`validate` 를 따로 둔 이유가 이것입니다. 검사를 `build_index` 안에 넣었다면 앞의 문서 몇 개는 이미 다시 넣은 뒤에 멈췄을 것입니다.

**바꿔 보기**
- `MIN_CHARS = 50` 을 `0` 으로 바꿔 검사를 꺼 보고 싶다면, 먼저 예제 6 까지 마친 뒤에 해 봅니다(지금은 나쁜 문서가 Lake 에 남아 있습니다).

## 예제 6 · 문서를 지우면 — 구멍을 찾고 prune 으로 메우기

나쁜 문서를 투입구에서 지우고 다시 돌리면 해결될까요? 「문서를 지우면 지식베이스는 어떻게 되나」를 먼저 예상해 보고 돌려 봅니다.

In [ ]:
os.remove("/content/inbox/empty.txt")
print("투입구에서 empty.txt 를 지웠습니다\n")

s = run_dag("rag_pipeline")
print()
print("Lake 에 있는 문서 →", sorted(os.listdir("/content/lake/raw-docs")))

***확인 포인트***: 투입구에서 지웠는데도 **`validate` 가 또 실패**합니다. Lake 의 목록에 `empty.txt` 가 그대로 있기 때문입니다.
`collect` 는 「투입구에 있는 것을 올린다」만 했지 「투입구에서 사라진 것을 내린다」는 하지 않았습니다. 이것이 처음 설계의 구멍입니다.

좋은 문서를 지웠을 때도 같습니다. Lake 와 지식베이스에 조각이 남아, 지운 문서의 내용으로 계속 답하게 됩니다. 두 곳을 고칩니다.

- `collect` — 투입구에 없는 문서는 Lake 에서도 내린다
- `prune`(새 태스크) — Lake 에 없는 문서의 조각을 지식베이스에서 지운다

In [ ]:
# rag_pipeline.py 를 고칩니다 — ① collect 가 Lake 를 투입구와 같게 맞추고 ② prune 태스크가 Lake 에 없는 문서의 조각을 지웁니다
from pathlib import Path

p = Path("/content/dags/rag_pipeline.py")
s = p.read_text()

s = s.replace(
    """    print(f"Lake 에 올린 문서 {len(names)}개 → {names}")""",
    """    gone = sorted(q.name for q in kb.LAKE.glob("*.txt") if q.name not in names)
    for name in gone:                       # 투입구에서 사라진 문서는 Lake 에서도 내립니다
        (kb.LAKE / name).unlink()
    print(f"Lake 에 올린 문서 {len(names)}개 → {names} · 내린 문서 {gone}")""")

s = s.replace(
    """def report(ti):""",
    """def prune():
    \"\"\"Lake 에 없는 문서의 조각을 지식베이스에서 지웁니다.\"\"\"
    keep = {q.name for q in kb.LAKE.glob("*.txt")}
    c = kb.client()
    try:
        gone = [name for name in kb.stat(c) if name not in keep]
        for name in gone:
            kb.remove_doc(c, name)
        print(f"지운 문서 → {gone}")
        return gone
    finally:
        c.close()


def report(ti):""")

s = s.replace("for f in (collect, validate, build_index, report)}", "for f in (collect, validate, build_index, prune, report)}")
s = s.replace('t["build_index"] >> t["report"]', 't["build_index"] >> t["prune"] >> t["report"]')
p.write_text(s)
print([l.strip() for l in s.splitlines() if ">>" in l][0])

In [ ]:
s = run_dag("rag_pipeline")
graph("rag_pipeline")

나쁜 문서가 내려가 다시 초록입니다. 이번에는 **좋은 문서 하나**를 투입구에서 지워 봅니다.

In [ ]:
os.remove("/content/inbox/minio_guide.txt")
print("투입구에서 minio_guide.txt 를 지웠습니다\n")

s = run_dag("rag_pipeline")
print()
kb("stat")
print()
kb("ask", "원본 문서는 어디에 보관하나요?")

***확인 포인트***: 고친 뒤 첫 실행에서 `collect` 가 「내린 문서 ['empty.txt']」를 찍고 다섯 태스크가 모두 success 입니다.
`minio_guide.txt` 를 지우고 돌리면 `prune` 이 「지운 문서 → ['minio_guide.txt']」를 찍고, 지식베이스는 **9개 · 문서 3개**가 됩니다. 「원본 문서 보관」 질문은 이제 근거가 없어 「찾지 못했습니다 (… 0.214 < 기준 0.4)」가 나옵니다.

지운 문서의 내용으로 계속 답하는 챗봇은 틀린 답보다 발견하기 어렵습니다. 「넣기」만큼 「빼기」를 설계해야 하는 이유입니다.

**바꿔 보기**
- 지운 문서를 되살립니다: `put("minio_guide.txt", DOCS["minio_guide.txt"])` 뒤 다시 돌려 12개 · 4개로 돌아오는지 확인합니다.
- 문서의 머리말 · 꼬리말을 떼는 `clean` 태스크를 `validate` 와 `build_index` 사이에 넣어 봅니다.

## 마무리
이 노트북은 「코랩으로 만드는 업무 챗봇」 16장의 예제입니다. 「바꿔 보기」에서 고친 것과 달라진 결과를 한두 줄로 적어 두면, 내 문서로 바꿀 때 길잡이가 됩니다.